# NBA Live Win Probability Dataset Builder

This notebook builds the play-by-play dataset used for NBA live win probability modeling.

It follows the same logic as the original working notebook, but is organized into clear sections and removes exploratory/debug cells.

In [ ]:
# If needed, uncomment and run once:
# !pip install pandas numpy requests tqdm

## 1. Imports and configuration

In [ ]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.nba.com/",
}

# Seasons use the start year:
# 2022 = 2022-23, 2023 = 2023-24, 2024 = 2024-25.
SEASONS = [2022, 2023, 2024]

BASE_PBP_PATH = Path("nba_win_prob_dataset.csv")
SCHEDULE_PATH = Path("schedule_with_tricodes.csv")
REST_PATH = Path("game_rest_features.csv")
TEAM_STATS_PATH = Path("team_rolling_stats.csv")
FINAL_PATH = Path("nba_win_prob_final.csv")
FINAL_V2_PATH = Path("nba_win_prob_final_v2.csv")

REQUEST_SLEEP_SECONDS = 0.6
CHECKPOINT_EVERY_GAMES = 100

# Set these to True when you want to rebuild from scratch.
RUN_PBP_PULL = True
RUN_PLAYOFF_PATCH = True

## 2. NBA API helpers

In [ ]:
def generate_game_ids(season_year, season_type="regular"):
    """Generate candidate NBA game IDs for a season.

    Regular-season IDs are generated as 002YYxxxxx.
    The first playoff pass uses the simple 004YYxxxxx pattern; a later patch
    adds the specific playoff series/game IDs that this simple pattern misses.
    """
    short_year = str(season_year + 1)[-2:]

    if season_type == "regular":
        prefix = f"002{short_year}"
        max_games = 1230
    else:
        prefix = f"004{short_year}"
        max_games = 105

    return [f"{prefix}{str(i).zfill(5)}" for i in range(1, max_games + 1)]


def try_get_pbp(game_id):
    """Return one game's live play-by-play as a DataFrame, or None if missing."""
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)

        if r.status_code != 200:
            return None

        data = r.json()
        actions = data["game"]["actions"]

        if not actions:
            return None

        return pd.DataFrame(actions)

    except Exception:
        return None


def parse_clock(clock_str):
    """Convert an NBA ISO clock string like PT11M42.00S into seconds."""
    match = re.match(r"PT(\d+)M([\d.]+)S", str(clock_str))

    if match:
        return int(match.group(1)) * 60 + float(match.group(2))

    return None

## 3. Convert raw play-by-play into game-state rows

In [ ]:
def build_game_state(raw_df, game_id, is_playoffs):
    """Clean one raw play-by-play table into model-ready game-state rows."""
    df = raw_df.copy()

    # The original working notebook inferred home/away team IDs from possession values.
    # This preserves that logic so the dataset is built the same way.
    possession_counts = df[df["possession"] != 0]["possession"].value_counts()
    if len(possession_counts) < 2:
        return None

    team_ids = possession_counts.index.tolist()
    home_team_id = team_ids[0]
    away_team_id = team_ids[1]

    action_types_keep = [
        "2pt", "3pt", "freethrow", "turnover", "foul",
        "rebound", "block", "steal", "timeout", "ejection"
    ]

    df = df[df["actionType"].isin(action_types_keep)].copy()
    if df.empty:
        return None

    # Some games/actions may not have every field.
    expected_cols = [
        "shotDistance", "area", "areaDetail", "shotResult", "isFieldGoal",
        "qualifiers", "foulPersonalTotal", "foulTechnicalTotal",
        "subType", "teamId", "teamTricode", "possession",
        "scoreHome", "scoreAway", "period", "clock", "personId",
        "playerName", "description", "x", "y"
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = None

    # Time
    df["seconds_in_period"] = df["clock"].apply(parse_clock)

    def seconds_remaining(row):
        period = row["period"]
        seconds = row["seconds_in_period"]

        if seconds is None:
            return None

        if period <= 4:
            return (4 - period) * 720 + seconds

        return max(0, 300 - seconds)

    df["seconds_remaining"] = df.apply(seconds_remaining, axis=1)
    df["is_overtime"] = (df["period"] > 4).astype(int)

    # Score
    df["scoreHome"] = pd.to_numeric(df["scoreHome"], errors="coerce").ffill().fillna(0)
    df["scoreAway"] = pd.to_numeric(df["scoreAway"], errors="coerce").ffill().fillna(0)
    df["score_diff"] = df["scoreHome"] - df["scoreAway"]
    df["momentum_5"] = df["score_diff"].diff(5)
    df["momentum_10"] = df["score_diff"].diff(10)

    # Possession
    df["home_possession"] = (df["possession"] == home_team_id).astype(int)

    # Fouls
    home_mask = df["teamId"] == home_team_id
    away_mask = df["teamId"] == away_team_id

    def propagate(series, index):
        return series.ffill().reindex(index).ffill().fillna(0)

    df["home_fouls"] = propagate(df[home_mask]["foulPersonalTotal"], df.index)
    df["away_fouls"] = propagate(df[away_mask]["foulPersonalTotal"], df.index)
    df["home_tech_fouls"] = propagate(df[home_mask]["foulTechnicalTotal"], df.index)
    df["away_tech_fouls"] = propagate(df[away_mask]["foulTechnicalTotal"], df.index)

    # Ejections
    df["home_ejections"] = 0
    df["away_ejections"] = 0

    ejections = df[df["actionType"] == "ejection"]
    for idx, row in ejections.iterrows():
        loc = df.index.get_loc(idx)

        if row["teamId"] == home_team_id:
            df.iloc[loc:, df.columns.get_loc("home_ejections")] += 1
        else:
            df.iloc[loc:, df.columns.get_loc("away_ejections")] += 1

    # Timeouts
    df["home_timeouts_used"] = 0
    df["away_timeouts_used"] = 0

    timeouts = df[df["actionType"] == "timeout"]
    for idx, row in timeouts.iterrows():
        loc = df.index.get_loc(idx)

        if row["teamId"] == home_team_id:
            df.iloc[loc:, df.columns.get_loc("home_timeouts_used")] += 1
        else:
            df.iloc[loc:, df.columns.get_loc("away_timeouts_used")] += 1

    # Shot context
    df["shot_distance"] = pd.to_numeric(df["shotDistance"], errors="coerce")
    df["shot_area"] = df["area"]
    df["shot_area_detail"] = df["areaDetail"]
    df["shot_result"] = df["shotResult"]
    df["is_field_goal"] = pd.to_numeric(df["isFieldGoal"], errors="coerce").fillna(0)
    df["shot_x"] = pd.to_numeric(df["x"], errors="coerce")
    df["shot_y"] = pd.to_numeric(df["y"], errors="coerce")

    def has_qualifier(q, tag):
        return int(tag in q) if isinstance(q, list) else 0

    df["is_fastbreak"] = df["qualifiers"].apply(lambda q: has_qualifier(q, "fastbreak"))
    df["is_painttouch"] = df["qualifiers"].apply(lambda q: has_qualifier(q, "pointsinthepaint"))
    df["is_fromturnover"] = df["qualifiers"].apply(lambda q: has_qualifier(q, "fromturnover"))

    # Free throw context
    df["freethrow_subtype"] = df.apply(
        lambda r: r["subType"] if r["actionType"] == "freethrow" else None,
        axis=1
    )

    # Player identity
    df["person_id"] = df["personId"]
    df["player_name"] = df["playerName"]
    df["description"] = df["description"]

    # Win label
    home_won = int(df["scoreHome"].iloc[-1] > df["scoreAway"].iloc[-1])

    # Metadata
    df["game_id"] = str(game_id).zfill(10)
    df["is_playoffs"] = int(is_playoffs)
    df["home_team_won"] = home_won
    df["home_team_id"] = home_team_id
    df["away_team_id"] = away_team_id

    keep = [
        "game_id", "is_playoffs", "home_team_won", "home_team_id", "away_team_id",
        "period", "is_overtime", "seconds_remaining",
        "scoreHome", "scoreAway", "score_diff", "momentum_5", "momentum_10",
        "home_possession", "actionType", "teamTricode",
        "home_fouls", "away_fouls", "home_tech_fouls", "away_tech_fouls",
        "home_ejections", "away_ejections", "home_timeouts_used", "away_timeouts_used",
        "shot_distance", "shot_area", "shot_area_detail", "shot_result",
        "is_field_goal", "shot_x", "shot_y",
        "is_fastbreak", "is_painttouch", "is_fromturnover",
        "freethrow_subtype",
        "person_id", "player_name", "description",
    ]

    return df[keep].reset_index(drop=True)

## 4. Pull regular-season play-by-play

In [ ]:
def run_pipeline(seasons, output_path=BASE_PBP_PATH, batch_size=CHECKPOINT_EVERY_GAMES):
    """Pull regular-season and simple playoff-ID play-by-play, with checkpoints."""
    output_path = Path(output_path)
    all_dfs = []
    done_ids = set()

    if output_path.exists():
        existing = pd.read_csv(output_path, low_memory=False)
        existing["game_id"] = existing["game_id"].astype(str).str.zfill(10)
        done_ids = set(existing["game_id"].astype(str).unique())
        all_dfs.append(existing)
        print(f"Resuming: {len(done_ids):,} games already processed.")

    games_processed = 0
    games_skipped = 0

    for season_year in seasons:
        for season_type, is_playoffs in [("regular", False), ("playoffs", True)]:
            label = (
                f"{season_year}-{str(season_year + 1)[-2:]} "
                f"{'Playoffs' if is_playoffs else 'Regular Season'}"
            )

            game_ids = generate_game_ids(season_year, season_type)
            game_ids = [gid for gid in game_ids if gid not in done_ids]

            print(f"\n{label}: trying {len(game_ids):,} candidate IDs")

            for game_id in tqdm(game_ids, desc=label):
                raw = try_get_pbp(game_id)

                if raw is None:
                    games_skipped += 1
                    time.sleep(REQUEST_SLEEP_SECONDS)
                    continue

                clean = build_game_state(raw, game_id, is_playoffs)

                if clean is not None:
                    all_dfs.append(clean)
                    done_ids.add(game_id)
                    games_processed += 1

                if games_processed > 0 and games_processed % batch_size == 0:
                    checkpoint = pd.concat(all_dfs, ignore_index=True)
                    checkpoint.to_csv(output_path, index=False)
                    print(
                        f"Checkpoint saved: {checkpoint['game_id'].nunique():,} games, "
                        f"{len(checkpoint):,} rows"
                    )

                time.sleep(REQUEST_SLEEP_SECONDS)

    if not all_dfs:
        print("No data collected.")
        return None

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df["game_id"] = final_df["game_id"].astype(str).str.zfill(10)
    final_df.to_csv(output_path, index=False)

    print(
        f"Done: {final_df['game_id'].nunique():,} games, "
        f"{len(final_df):,} rows saved to {output_path}"
    )
    print(f"Skipped/missing candidate IDs: {games_skipped:,}")

    return final_df


if RUN_PBP_PULL:
    df_full = run_pipeline(SEASONS, output_path=BASE_PBP_PATH)
else:
    df_full = pd.read_csv(BASE_PBP_PATH, low_memory=False)
    df_full["game_id"] = df_full["game_id"].astype(str).str.zfill(10)
    print(f"Loaded existing play-by-play dataset: {df_full.shape}")

## 5. Patch playoff games

In [ ]:
def generate_playoff_ids(season_year):
    """Generate playoff IDs using the series-code pattern found in the original notebook."""
    short_year = str(season_year + 1)[-2:]
    ids = []

    # Round/series codes used by NBA CDN playoff IDs.
    series_codes = [
        10, 11, 12, 13, 14, 15, 16, 17,
        20, 21, 22, 23,
        30, 31,
        40,
    ]

    for series_code in series_codes:
        for game in range(1, 8):
            ids.append(f"004{short_year}00{series_code}{game}")

    return ids


def run_playoff_patch(seasons, output_path=BASE_PBP_PATH):
    """Add playoff games missed by the simple playoff-ID pass."""
    output_path = Path(output_path)

    existing = pd.read_csv(output_path, low_memory=False)
    existing["game_id"] = existing["game_id"].astype(str).str.zfill(10)

    done_ids = set(existing["game_id"].astype(str).unique())
    all_dfs = [existing]
    games_added = 0

    for season_year in seasons:
        short_year = str(season_year + 1)[-2:]
        game_ids = generate_playoff_ids(season_year)
        game_ids = [gid for gid in game_ids if gid not in done_ids]

        print(f"\n{season_year}-{short_year} playoffs: trying {len(game_ids):,} candidate IDs")

        for game_id in tqdm(game_ids):
            raw = try_get_pbp(game_id)

            if raw is None:
                time.sleep(REQUEST_SLEEP_SECONDS)
                continue

            clean = build_game_state(raw, game_id, is_playoffs=True)

            if clean is not None:
                all_dfs.append(clean)
                done_ids.add(game_id)
                games_added += 1

            if games_added > 0 and games_added % 25 == 0:
                checkpoint = pd.concat(all_dfs, ignore_index=True)
                checkpoint.to_csv(output_path, index=False)
                print(f"Checkpoint: {games_added:,} playoff games added")

            time.sleep(REQUEST_SLEEP_SECONDS)

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df["game_id"] = final_df["game_id"].astype(str).str.zfill(10)
    final_df.to_csv(output_path, index=False)

    print(f"Done: added {games_added:,} playoff games")
    print(f"Total: {final_df['game_id'].nunique():,} games, {len(final_df):,} rows")

    return final_df


if RUN_PLAYOFF_PATCH:
    # This matches the original notebook: patch completed 2022-23 and 2023-24 playoffs.
    df_patched = run_playoff_patch([2022, 2023], output_path=BASE_PBP_PATH)
else:
    df_patched = pd.read_csv(BASE_PBP_PATH, low_memory=False)
    df_patched["game_id"] = df_patched["game_id"].astype(str).str.zfill(10)
    print(f"Loaded patched play-by-play dataset: {df_patched.shape}")

## 6. Build schedule and rest features

In [ ]:
def parse_schedule_old(year):
    """Parse NBA schedule from the older data.nba.com endpoint."""
    url = f"https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/{year}/league/00_full_schedule.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    rows = []

    for month in r.json()["lscd"]:
        for game in month["mscd"]["g"]:
            gcode = game["gcode"]
            rows.append({
                "game_id": str(game["gid"]).zfill(10),
                "game_date": pd.to_datetime(game["gdte"]),
                "home_tricode": gcode[9:][3:],
                "away_tricode": gcode[9:][:3],
                "season_year": int(year),
            })

    return pd.DataFrame(rows)


def parse_schedule_current():
    """Parse current NBA schedule from the CDN schedule endpoint."""
    url = "https://cdn.nba.com/static/json/staticData/scheduleLeagueV2.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()

    rows = []

    for date_entry in r.json()["leagueSchedule"]["gameDates"]:
        for game in date_entry["games"]:
            gcode = game["gameCode"]
            date_str = gcode[:8]
            teams = gcode[9:]
            game_id = str(game["gameId"]).zfill(10)

            # Original notebook kept current regular-season rows only.
            if not game_id.startswith("002"):
                continue

            rows.append({
                "game_id": game_id,
                "game_date": pd.to_datetime(date_str, format="%Y%m%d"),
                "home_tricode": teams[3:],
                "away_tricode": teams[:3],
            })

    return pd.DataFrame(rows)


def build_full_schedule():
    """Build the schedule table used to map games to home/away tricodes."""
    print("Fetching schedules...")

    schedules = [
        parse_schedule_old("2022"),
        parse_schedule_old("2023"),
        parse_schedule_old("2024"),
        parse_schedule_current(),
    ]

    full_schedule = pd.concat(schedules, ignore_index=True)
    full_schedule["game_id"] = full_schedule["game_id"].astype(str).str.zfill(10)
    full_schedule = full_schedule[full_schedule["game_id"].str.startswith(("002", "004"))]
    full_schedule = full_schedule.drop_duplicates("game_id")

    full_schedule.to_csv(SCHEDULE_PATH, index=False)

    print(f"Schedule rows: {len(full_schedule):,}")
    print(full_schedule["game_id"].str[:4].value_counts().sort_index())

    return full_schedule


full_schedule = build_full_schedule()

In [ ]:
def build_rest_features(schedule):
    """Compute rest days, back-to-backs, and three-in-four indicators."""
    schedule = schedule.copy()
    schedule["game_id"] = schedule["game_id"].astype(str).str.zfill(10)
    schedule["game_date"] = pd.to_datetime(schedule["game_date"])

    home = schedule[["game_id", "game_date", "home_tricode"]].rename(
        columns={"home_tricode": "tricode"}
    )
    home["is_home"] = 1

    away = schedule[["game_id", "game_date", "away_tricode"]].rename(
        columns={"away_tricode": "tricode"}
    )
    away["is_home"] = 0

    team_games = (
        pd.concat([home, away], ignore_index=True)
        .sort_values(["tricode", "game_date"])
        .reset_index(drop=True)
    )

    team_games["prev_game_date"] = team_games.groupby("tricode")["game_date"].shift(1)
    team_games["rest_days"] = (team_games["game_date"] - team_games["prev_game_date"]).dt.days
    team_games["is_back_to_back"] = (team_games["rest_days"] == 1).astype(int)
    team_games["is_3_in_4"] = (team_games["rest_days"] <= 2).astype(int)

    home_rest = team_games[team_games["is_home"] == 1][
        ["game_id", "game_date", "rest_days", "is_back_to_back", "is_3_in_4"]
    ].copy()
    home_rest.columns = ["game_id", "game_date", "home_rest_days", "home_b2b", "home_3in4"]

    away_rest = team_games[team_games["is_home"] == 0][
        ["game_id", "rest_days", "is_back_to_back", "is_3_in_4"]
    ].copy()
    away_rest.columns = ["game_id", "away_rest_days", "away_b2b", "away_3in4"]

    rest_df = home_rest.merge(away_rest, on="game_id")
    rest_df["game_id"] = rest_df["game_id"].astype(str).str.zfill(10)
    rest_df.to_csv(REST_PATH, index=False)

    print(f"Rest features saved for {len(rest_df):,} games.")
    print(rest_df[["home_rest_days", "away_rest_days"]].isna().sum())

    return rest_df


rest_df_full = build_rest_features(full_schedule)

## 7. Compute team-level rolling statistics

In [ ]:
def compute_game_team_stats(df):
    """Aggregate cleaned play-by-play into one row per team per game."""
    rows = []

    for (game_id, tricode), grp in df.groupby(["game_id", "teamTricode"]):
        if pd.isna(tricode) or tricode == "":
            continue

        shots_2pt = grp[grp["actionType"] == "2pt"]
        shots_3pt = grp[grp["actionType"] == "3pt"]
        fts = grp[grp["actionType"] == "freethrow"]
        turnovers = grp[grp["actionType"] == "turnover"]
        rebounds = grp[grp["actionType"] == "rebound"]

        fgm_2 = (shots_2pt["shot_result"] == "Made").sum()
        fga_2 = len(shots_2pt)
        fgm_3 = (shots_3pt["shot_result"] == "Made").sum()
        fga_3 = len(shots_3pt)
        fgm = fgm_2 + fgm_3
        fga = fga_2 + fga_3
        ftm = (fts["shot_result"] == "Made").sum()
        fta = len(fts)
        tov = len(turnovers)
        oreb = rebounds["description"].str.lower().str.contains("off:", na=False).sum()
        dreb = rebounds["description"].str.lower().str.contains("def:", na=False).sum()
        pts = fgm_2 * 2 + fgm_3 * 3 + ftm

        efg = (fgm + 0.5 * fgm_3) / fga if fga > 0 else None
        possessions = fga + 0.44 * fta + tov
        tov_pct = tov / possessions if possessions > 0 else None
        ft_rate = fta / fga if fga > 0 else None

        rows.append({
            "game_id": game_id,
            "tricode": tricode,
            "pts": pts,
            "fgm": fgm,
            "fga": fga,
            "fgm_3": fgm_3,
            "fga_3": fga_3,
            "ftm": ftm,
            "fta": fta,
            "tov": tov,
            "oreb": oreb,
            "dreb": dreb,
            "efg": efg,
            "tov_pct": tov_pct,
            "ft_rate": ft_rate,
            "possessions": possessions,
        })

    return pd.DataFrame(rows)


def add_opponent_and_rating_stats(game_team_stats):
    """Add opponent box-score pieces and simple offensive/defensive ratings."""
    opp = game_team_stats[["game_id", "tricode", "pts", "possessions", "dreb", "oreb"]].copy()
    opp.columns = ["game_id", "opp_tricode", "opp_pts", "opp_possessions", "opp_dreb", "opp_oreb"]

    out = game_team_stats.merge(opp, on="game_id", how="left")
    out = out[out["tricode"] != out["opp_tricode"]].copy()

    out["oreb_pct"] = out["oreb"] / (out["oreb"] + out["opp_dreb"]).replace(0, np.nan)
    out["off_rtg"] = (out["pts"] / out["possessions"] * 100).replace([np.inf, -np.inf], np.nan)
    out["def_rtg"] = (out["opp_pts"] / out["opp_possessions"] * 100).replace([np.inf, -np.inf], np.nan)
    out["net_rtg"] = out["off_rtg"] - out["def_rtg"]
    out["won"] = (out["pts"] > out["opp_pts"]).astype(int)

    return out


def add_rolling_features(df, cols, window, label):
    """Add shifted rolling averages by team to avoid current-game leakage."""
    out = df.copy()

    for col in cols:
        out[f"{col}_last{label}"] = (
            out.groupby("tricode")[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )

    return out


print("Loading play-by-play...")
pbp = pd.read_csv(BASE_PBP_PATH, low_memory=False)
pbp["game_id"] = pbp["game_id"].astype(str).str.zfill(10)

print("Computing per-game team stats...")
game_team_stats = compute_game_team_stats(pbp)
game_team_stats = add_opponent_and_rating_stats(game_team_stats)

# Attach dates/home-away mapping.
team_dates = pd.concat([
    full_schedule[["game_id", "game_date", "home_tricode"]].rename(columns={"home_tricode": "tricode"}),
    full_schedule[["game_id", "game_date", "away_tricode"]].rename(columns={"away_tricode": "tricode"}),
], ignore_index=True).drop_duplicates()

team_dates["game_id"] = team_dates["game_id"].astype(str).str.zfill(10)

game_team_stats["game_id"] = game_team_stats["game_id"].astype(str).str.zfill(10)
game_team_stats = game_team_stats.merge(team_dates, on=["game_id", "tricode"], how="left")

# The original notebook ultimately sorted by team/date before rolling.
game_team_stats = game_team_stats.sort_values(["tricode", "game_date"]).reset_index(drop=True)

stat_cols = [
    "efg", "tov_pct", "ft_rate", "oreb_pct",
    "off_rtg", "def_rtg", "net_rtg", "won", "pts"
]

print("Computing rolling features...")
game_team_stats = add_rolling_features(game_team_stats, stat_cols, 10, "10")
game_team_stats = add_rolling_features(game_team_stats, stat_cols, 20, "20")
game_team_stats = add_rolling_features(game_team_stats, stat_cols, 82, "season")

game_team_stats.to_csv(TEAM_STATS_PATH, index=False)

print(f"Team stats saved: {game_team_stats.shape}")
print(f"Missing game dates: {game_team_stats['game_date'].isna().sum():,}")

## 8. Join team strength, rest, and play-by-play

In [ ]:
def build_home_away_team_stats(team_stats, schedule):
    """Split team rolling stats into home_* and away_* rows by game."""
    team_stats = team_stats.copy()
    schedule = schedule.copy()

    team_stats["game_id"] = team_stats["game_id"].astype(str).str.zfill(10)
    schedule["game_id"] = schedule["game_id"].astype(str).str.zfill(10)

    rolling_cols = [
        c for c in team_stats.columns
        if any(c.endswith(suffix) for suffix in ["_last10", "_last20", "_lastseason"])
    ]

    id_cols = [
        "game_id", "tricode", "off_rtg", "def_rtg", "net_rtg", "efg",
        "tov_pct", "ft_rate", "oreb_pct", "won", "pts"
    ]

    keep_cols = id_cols + rolling_cols
    home_map = schedule[["game_id", "home_tricode", "away_tricode"]].copy()

    team_stats_full = team_stats[keep_cols].merge(home_map, on="game_id", how="left")

    home_stats = team_stats_full[
        team_stats_full["tricode"] == team_stats_full["home_tricode"]
    ].copy()

    away_stats = team_stats_full[
        team_stats_full["tricode"] == team_stats_full["away_tricode"]
    ].copy()

    home_stats = home_stats.drop(columns=["tricode", "home_tricode", "away_tricode"])
    away_stats = away_stats.drop(columns=["tricode", "home_tricode", "away_tricode"])

    home_stats.columns = ["game_id"] + [
        f"home_{c}" for c in home_stats.columns if c != "game_id"
    ]

    away_stats.columns = ["game_id"] + [
        f"away_{c}" for c in away_stats.columns if c != "game_id"
    ]

    return home_stats, away_stats


def add_diff_features(df):
    """Add home-minus-away rolling stat differences and date features."""
    out = df.copy()

    diff_base = ["off_rtg", "def_rtg", "net_rtg", "efg", "tov_pct", "ft_rate", "oreb_pct"]

    for window in ["last10", "last20", "lastseason"]:
        for stat in diff_base:
            home_col = f"home_{stat}_{window}"
            away_col = f"away_{stat}_{window}"

            if home_col in out.columns and away_col in out.columns:
                out[f"diff_{stat}_{window}"] = out[home_col] - out[away_col]

    out["game_date"] = pd.to_datetime(out["game_date"])
    out["day_of_week"] = out["game_date"].dt.dayofweek
    out["is_weekend"] = out["day_of_week"].isin([4, 5, 6]).astype(int)

    return out


pbp = pd.read_csv(BASE_PBP_PATH, low_memory=False)
pbp["game_id"] = pbp["game_id"].astype(str).str.zfill(10)

# This mirrors the original notebook's choice to exclude current-season playoff rows.
pbp = pbp[~pbp["game_id"].str.startswith("0042500")].copy()

team_stats = pd.read_csv(TEAM_STATS_PATH, low_memory=False)
team_stats["game_id"] = team_stats["game_id"].astype(str).str.zfill(10)

rest_df = pd.read_csv(REST_PATH, parse_dates=["game_date"])
rest_df["game_id"] = rest_df["game_id"].astype(str).str.zfill(10)

home_stats, away_stats = build_home_away_team_stats(team_stats, full_schedule)

print(f"Home stats: {home_stats.shape}")
print(f"Away stats: {away_stats.shape}")

df_final = pbp.copy()
df_final = df_final.merge(home_stats, on="game_id", how="left")
df_final = df_final.merge(away_stats, on="game_id", how="left")

rest_cols = [
    "game_id", "game_date",
    "home_rest_days", "home_b2b", "home_3in4",
    "away_rest_days", "away_b2b", "away_3in4",
]

df_final = df_final.merge(rest_df[rest_cols], on="game_id", how="left")
df_final = add_diff_features(df_final)

df_final.to_csv(FINAL_PATH, index=False)

print(f"Final dataset saved: {df_final.shape}")
print(df_final[[
    "home_net_rtg_last10", "away_net_rtg_last10",
    "home_rest_days", "away_rest_days"
]].isna().sum())

## 9. Re-join rest features in chunks

In [ ]:
# This chunked pass matches the final step in the original notebook.
# It is useful when the full final file is too large to comfortably rewrite at once.

rest_df_full = pd.read_csv(REST_PATH, parse_dates=["game_date"])
rest_df_full["game_id"] = rest_df_full["game_id"].astype(str).str.zfill(10)

chunk_size = 100_000
first_chunk = True

rest_cols = [
    "game_id", "game_date",
    "home_rest_days", "home_b2b", "home_3in4",
    "away_rest_days", "away_b2b", "away_3in4",
]

for chunk in pd.read_csv(FINAL_PATH, chunksize=chunk_size, low_memory=False):
    chunk["game_id"] = chunk["game_id"].astype(str).str.zfill(10)

    chunk = chunk.drop(
        columns=[
            "home_rest_days", "home_b2b", "home_3in4",
            "away_rest_days", "away_b2b", "away_3in4",
            "game_date", "day_of_week", "is_weekend",
        ],
        errors="ignore",
    )

    chunk = chunk.merge(rest_df_full[rest_cols], on="game_id", how="left")

    chunk["game_date"] = pd.to_datetime(chunk["game_date"])
    chunk["day_of_week"] = chunk["game_date"].dt.dayofweek
    chunk["is_weekend"] = chunk["day_of_week"].isin([4, 5, 6]).astype(int)

    chunk.to_csv(
        FINAL_V2_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    first_chunk = False
    print(".", end="", flush=True)

print(f"\nDone. Saved to {FINAL_V2_PATH}")

sample = pd.read_csv(FINAL_V2_PATH, nrows=1000)
print(f"Columns: {len(sample.columns)}")
print(sample[["home_rest_days", "away_rest_days"]].isna().sum().to_dict())

## 10. Create slim modeling file

In [ ]:
# Optional: create a smaller file for modeling on machines with limited RAM.

SLIM_PATH = Path("nba_win_prob_slim.csv")

slim_cols = [
    "game_id", "is_playoffs", "home_team_won",
    "period", "is_overtime", "seconds_remaining", "score_diff",
    "momentum_5", "momentum_10", "home_possession", "actionType",
    "home_fouls", "away_fouls", "home_tech_fouls", "away_tech_fouls",
    "home_ejections", "away_ejections", "home_timeouts_used", "away_timeouts_used",
    "shot_distance", "shot_area", "shot_result", "is_field_goal",
    "shot_x", "shot_y", "is_fastbreak", "is_painttouch", "is_fromturnover",
    "home_net_rtg_last10", "home_net_rtg_last20", "home_net_rtg_lastseason",
    "home_off_rtg_last10", "home_def_rtg_last10", "home_efg_last10",
    "home_tov_pct_last10", "home_oreb_pct_last10",
    "home_off_rtg_lastseason", "home_def_rtg_lastseason",
    "home_won_last10", "home_won_lastseason",
    "away_net_rtg_last10", "away_net_rtg_last20", "away_net_rtg_lastseason",
    "away_off_rtg_last10", "away_def_rtg_last10", "away_efg_last10",
    "away_tov_pct_last10", "away_oreb_pct_last10",
    "away_off_rtg_lastseason", "away_def_rtg_lastseason",
    "away_won_last10", "away_won_lastseason",
    "game_date", "home_rest_days", "away_rest_days",
    "home_b2b", "away_b2b", "home_3in4", "away_3in4",
    "diff_net_rtg_last10", "diff_net_rtg_last20", "diff_net_rtg_lastseason",
    "diff_off_rtg_last10", "diff_def_rtg_last10",
    "day_of_week", "is_weekend",
]

sample = pd.read_csv(FINAL_V2_PATH, nrows=1)
available_slim_cols = [c for c in slim_cols if c in sample.columns]

first_chunk = True
for chunk in pd.read_csv(FINAL_V2_PATH, chunksize=100_000, low_memory=False):
    chunk["game_id"] = chunk["game_id"].astype(str).str.zfill(10)

    chunk[available_slim_cols].to_csv(
        SLIM_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    first_chunk = False
    print(".", end="", flush=True)

print(f"\nSlim file saved to {SLIM_PATH}")

## 11. Final validation

In [ ]:
final_sample = pd.read_csv(FINAL_V2_PATH, nrows=100_000, low_memory=False)
final_sample["game_id"] = final_sample["game_id"].astype(str).str.zfill(10)

print(f"Sample shape: {final_sample.shape}")
print(f"Sample games: {final_sample['game_id'].nunique():,}")

key_cols = [
    "game_id", "home_team_won", "period", "seconds_remaining", "score_diff",
    "home_net_rtg_lastseason", "away_net_rtg_lastseason",
    "home_rest_days", "away_rest_days",
]

existing_key_cols = [c for c in key_cols if c in final_sample.columns]

print("\nNull check:")
print(final_sample[existing_key_cols].isna().sum())

print("\nFirst rows:")
display(final_sample[existing_key_cols].head())